In [14]:
import pandas as pd

df = pd.read_csv('long_092326_senspec.csv').set_index('peptide')

In [15]:
lasv = df.loc[df['species'] == 'Mammarenavirus lassaense']

In [16]:
lasv = lasv.reset_index()

In [17]:
lasv.head(2)

,peptide,individual,z_score,rpk,sequence,species,genus,family,protein_std,accession,cohort,category,reactive
0,AAA46284.1|?|Mammarenavirus_lassaense|?|?|?|?|...,C-109-3_4_E7,4.832718,5.063550,TERPLSSGVYMGNLSSQQLDQRRALLNMIGMTGVSGGGKGASNGIV,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,nucleoprotein,AAA46284.1,SL,Household contact,True
1,AAO59509.1|polymerase|Mammarenavirus_lassaense...,C-109-3_4_E7,1.470043,1.164227,DNRSDHYEEIIALCHQGINNKLTAHEVKLQIEEEYQVFRNRLRGGE,Mammarenavirus lassaense,Mammarenavirus,Arenaviridae,polymerase,AAO59509.1,SL,Household contact,False


In [18]:
peps_mean = lasv.groupby(['cohort', 'protein_std', 'peptide', 'sequence'])['reactive'].mean().sort_values(ascending=False).reset_index()

In [19]:
peps_mean['tile_num'] = peps_mean['peptide'].str.split('_').str[-1]

In [20]:
peps_mean.head(1)

,cohort,protein_std,peptide,sequence,reactive,tile_num
0,SL,polymerase,AIT17291.1|polymerase|Mammarenavirus_lassaense...,YKVQQAMSNLVLGSGQHKDGVDKADLDEILLDGGASIYFDQLRETV,0.536524,39


## now for each tile_num, I want the most seroprevalent peptide

In [21]:
top_peps_max = peps_mean.loc[
    peps_mean.groupby(['protein_std', 'tile_num'])['reactive'].idxmax(), 
    ['protein_std', 'tile_num', 'peptide', 'sequence', 'reactive']
].reset_index(drop=True).sort_values(by='reactive', ascending=False)
#want the row index where reactive is highest

In [22]:
top_peps_max = top_peps_max.rename(columns={'reactive': 'fraction_reactive'})

In [23]:
top_peps_max['fraction_reactive'] = top_peps_max['fraction_reactive'].round(2)

In [24]:
top_peps_max.to_csv('most_seroprevalent_LASV_peptides_per_tile_num.csv', index=False)

In [25]:
lasv.loc[(lasv['peptide'].isin(top_peps_max['peptide'].to_list())) & (lasv['reactive']==True)].to_csv('most_seroprevalent_LASV_peptides_per_tile_num_reactive_w_RPK_zscore.csv', index=False)